# ocean: 2D

2D ocean data

```{dropdown}  **Branding suffix of datasets**
Datasets with the folloing branding suffix are shown in this page: \
[time]-[vertical]-[horizontal]-[domain]
```
* tavg-u-hxy-sea
* tavg-u-hxy-ifs
* tmax-u-hxy-sea
* tmin-u-hxy-sea
* tavg-d300m-hxy-sea
* tavg-d700m-hxy-sea
* tavg-d2000m-hxy-sea
    - tavg: time average
    - u: ocean surface
    - d300m: layer at 300m depth
    - d700m: layer at 700m depth
    - d2000m: layer at 2000m depth
    - hxy: horizontal curvlinear grids
    - sea: ocean domain

In [ ]:
## Import libraries
import glob
import logging
import os

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmocean
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from IPython.display import HTML, display
from matplotlib.colors import BoundaryNorm, ListedColormap
from pathlib import Path
from pyprojroot import here
from xarray.coders import CFDatetimeCoder

from cmip7validate.utils import read_compound_names, read_variables

logging.basicConfig(
    filename="execute.log",
    filemode="w",
    level=logging.INFO,
    format="%(asctime)s %(message)s",
    force=True,
)
log = logging.getLogger("ocean_2D")
log.info("Starting ocean_2D.ipynb")

# Get project root
root_path = Path(here())

In [ ]:
# load parameters
import yaml

with open(root_path.joinpath('params.yml'), 'r') as f:
    param = yaml.safe_load(f)
    cmorout = param['datasets']['cmorout']
    source_id = param['datasets']['source_id']
    experiment_id = param['datasets']['experiment_id']
    variant_label = param['datasets']['variant_label']
    version = param['datasets']['version']
    cacheroot = param['cache']['root']
    cacheoverwrite = param['cache']['overwrite']

In [ ]:
# grid label
grid_label = 'g202'

# data path
data_path = Path(cmorout).joinpath(source_id, experiment_id, variant_label, version)

# cache path
cacheroot_climo = Path(cacheroot).joinpath(source_id, experiment_id, variant_label, version, 'climo')
cacheroot_ts = Path(cacheroot).joinpath(source_id, experiment_id, variant_label, version, 'ts')
cacheroot_climo.mkdir(parents=True, exist_ok=True)
cacheroot_ts.mkdir(parents=True, exist_ok=True)

# load grid
grid_file = data_path.joinpath(f"areacello_ti-u-hxy-u_fx_glb_{grid_label}_{source_id}_{experiment_id}_{variant_label}.nc")
if grid_file.exists():
    with xr.open_dataset(grid_file) as ds:
        parea = ds['areacello']
else:
    grid_file = root_path.joinpath('assets/grid.nc')
    with xr.open_dataset(grid_file) as ds:
        parea = ds['parea']
        parea = parea.rename({'y': 'j', 'x': 'i'})

grid_file = data_path.joinpath(f"sftof_ti-u-hxy-u_fx_glb_{grid_label}_{source_id}_{experiment_id}_{variant_label}.nc")
if grid_file.exists():
    with xr.open_dataset(grid_file) as ds:
        pmask = ds['sftof']
else:
    grid_file = root_path.joinpath('assets/grid.nc')
    with xr.open_dataset(grid_file) as ds:
        pmask = ds['pmask']
        pmask = pmask.rename({'y': 'j', 'x': 'i'})

# load methods for plotting and set defaults
methods =read_variables(root_path.joinpath('assets/methods.txt'))

---
```{dropdown}  **List of datasets:** 
List of compound names of datasets that needs to be cmorized. \
Those datasets which are not cmorized have no link.
```

In [ ]:
# load compound names
grid_label = 'g???'
coords = ('tavg-u-hxy-sea', 'tavg-u-hxy-ifs', 'tmax-u-hxy-sea', 'tmin-u-hxy-sea', 'tavg-d300m-hxy-sea', 'tavg-d700m-hxy-sea', 'tavg-d2000m-hxy-sea')
cnames = read_compound_names(root_path.joinpath('assets/variables.nml'))
# examples of compound names:
# cnames = ['ocean.tos.tavg-u-hxy-sea.mon.glb', 'ocean.ficeberg.tavg-u-hxy-sea.mon.glb']
for cname in cnames:
    realm = cname.split('.')[0]
    var = cname.split('.')[1]
    coord = cname.split('.')[2]
    freq = cname.split('.')[3]
    region = cname.split('.')[4]
    
    if realm != 'ocean' or coord not in coords:
        continue
    else:
        file_pattern = f"{var}_{coord}_{freq}_{region}_{grid_label}_{source_id}_{experiment_id}_{variant_label}_*.nc"
        if not sorted(data_path.glob(file_pattern)):
            print(cname)
            continue
        else:
            display(HTML(f'<a href="#{cname}">{cname}</a>'))

---
```{dropdown}  **Dataset validated:**
Each validated dataset has some text information and some plots (2D and/or 1D)
```

In [ ]:
import time
time0 = time.time()

# loop through compound names and plot
for cname in cnames:
#    mth_vert = 'mean'
    mth_ts = 'mean'
    mth_cmap = 'mpl.colormaps["viridis"]'
    if cname not in methods:
        print(f"{cname} not found in methods.txt, using default methods for plotting.")
    else:
        if methods[cname] is not None:
            if 'timeseries' in methods[cname]:
                mth_ts = methods[cname]['timeseries']

            if 'cmap' in methods[cname]:
                mth_cmap = methods[cname]['cmap']

    realm = cname.split('.')[0]
    var = cname.split('.')[1]
    coord = cname.split('.')[2]
    freq = cname.split('.')[3]
    region = cname.split('.')[4]

    if realm != 'ocean' or coord not in coords:
        continue

    file_pattern = f"{var}_{coord}_{freq}_{region}_{grid_label}_{source_id}_{experiment_id}_{variant_label}_*.nc"
    data_files = sorted(data_path.glob(file_pattern))

    if data_files:
        # read lat/lon
        with xr.open_dataset(data_files[0]) as ds:
            lat = ds['latitude']
            lon = ds['longitude']

        # remove the last row of grid info if the data has not the last row
        if lat.sizes['j'] == 384:
            parea = parea.isel(j=slice(0, 384))
            pmask = pmask.isel(j=slice(0, 384))

        #time0 = time.time()
        log.info(f"{cname}")
        total = None
        count = None
        data2d = None
        data1d = None
        for data_file in data_files:
            #print(data_file)
            # read/cacluate 2d time mean
            climo_file = cacheroot_climo.joinpath(data_file.name)
            if climo_file.is_file() and not cacheoverwrite:
                with xr.open_dataset(climo_file) as ds:
                    log.info(f"Reading climo cache for {data_file.name}")
                    daclimo = ds[var]
                    ntimes = ds.sizes['time']
            else:
               with xr.open_dataset(data_file) as ds:
                    da = ds[var]
                    ntimes = ds.sizes['time']
                    daclimo = da.mean(dim='time',keep_attrs=True)

                    # save climo (2D time mean) and ntimes to cache file
                    log.info(f"Calculate climo for {data_file.name}")
                    ds_climo = daclimo.to_dataset(name=var)
                    ds_climo['time'] = ds['time']
                    ds_climo.to_netcdf(climo_file, unlimited_dims=["time"])
                    log.info(f"Saved     climo to {climo_file.name}")

            total = daclimo*ntimes if total is None else total + daclimo*ntimes
            count = ntimes if count is None else count + ntimes
           
            # read/cacluate 1d time series
            ts_file = cacheroot_ts.joinpath(data_file.name)
            if ts_file.is_file() and not cacheoverwrite:
                with xr.open_dataset(ts_file) as ds:
                    log.info(f"Reading    TS cache for {data_file.name}")
                    dats = ds[var]
            else:
               with xr.open_dataset(data_file) as ds:
                    da = ds[var]
                    # Calculate 1D timeseries
                    if mth_ts == 'mean':
                        dats = da.weighted(parea).mean(dim=['j', 'i'], keep_attrs=True).compute()
                    elif mth_ts == 'sum':
                        dats = da.weighted(parea).sum(dim=['j', 'i'], keep_attrs=True).compute()
                    else:
                        raise ValueError(f'Unsupported timeseries method: {mth_ts}')
                    dats.attrs = da.attrs.copy()

                    # save ts (1D time series) to cache file
                    log.info(f"Calculate TS for {data_file.name}")
                    ds_ts = dats.to_dataset(name=var)
                    ds_ts['time'] = ds['time']
                    ds_ts.to_netcdf(ts_file, unlimited_dims=["time"])
                    log.info(f"Saved     TS to {ts_file.name}")
                    
            data1d = dats if data1d is None else xr.concat([data1d, dats], dim="time")
            
        data2d =  total / count
        data2d.attrs = daclimo.attrs.copy()

    else:
        continue

    display(HTML(f'<div id="{cname}"></div>'))
    print(f'\033[1m{cname}\033[0m')
    print(f'long name: {data2d.long_name} ({data2d.units})')
    print(f'timeseries aggregation method for 2D->1D plot: {mth_ts}')
    print(f'original_name: {data2d.attrs["original_name"]} -> {var}')
    if 'history' in data2d.attrs:
        print(f'history: {data2d.attrs["history"]}')
    if 'comment' in data2d.attrs:
        print(f'comment: {data2d.attrs["comment"]}')

    # #Create figure
    fig = plt.figure(figsize=(16, 3), dpi=96)
    gs = fig.add_gridspec(nrows=1, ncols=2, width_ratios=[2, 1]) 
    proj = ccrs.PlateCarree(central_longitude=90.0)
    # rescale
    vmin, vmax = data2d.quantile([0.001, 0.999], dim=None).values
    if mth_cmap == 'cmocean.cm.balance':
        vmax = max(abs(vmin),abs(vmax))
        vmin = -vmax
    # Define the discrete levels (boundaries) and the corresponding colors
    levels = np.linspace(vmin, vmax, 21)
    n_levels = 20
    colors = eval(mth_cmap)(np.linspace(0,1,n_levels))
    cmap = ListedColormap(colors)

    # Create a BoundaryNorm to map data to discrete color indices
    norm = BoundaryNorm(levels, ncolors=len(cmap.colors), clip=False)

    # 2D map
    ax1 = fig.add_subplot(gs[0], projection=proj)
    if vmin == vmax:
        pm = ax1.pcolormesh(lon, lat, data2d, cmap=cmap,
                            transform=ccrs.PlateCarree(), shading='auto', rasterized=True)
    else:
        pm = ax1.pcolormesh(lon, lat, data2d, cmap=cmap, norm=norm,
                            transform=ccrs.PlateCarree(), shading='auto', rasterized=True)

    gl = ax1.gridlines(ylocs=range(-90, 90, 30), draw_labels=True)
    gl.ylocator = mpl.ticker.FixedLocator(range(-90,90,30))

    # Add colorbar
    cb = plt.colorbar(pm, ax=ax1, fraction=0.4, shrink=0.8, label='[yr]', extend='both')
    cb.set_label(label=data2d.units, size=14)
    cb.ax.tick_params(labelsize=12)
    plt.tight_layout()
        
    # Add map features
    ax1.coastlines(resolution='110m')
    ax1.add_feature(cfeature.LAND, facecolor='lightgray')
    plt.suptitle(data2d.long_name, y=1.1)

    # 1D timeseries plot
    ax2 = fig.add_subplot(gs[1])
    if data1d.sizes["time"] < 2:
        ax2.text(0.1, 0.8, (f"global {mth_ts} value: {data1d.values[0]} {data1d.units}"))
        ax2.text(0.1, 0.6, ("less than 2 time steps."))
        ax2.text(0.1, 0.4, ("skipping timeseries plot."))
    else:
        data1d.plot(ax=ax2)

    plt.show()

    del data2d, data1d
    del fig, gs, ax1, ax2, pm, cb, gl
    log.info(f"time cost: {(time.time()-time0)/60:.1f} minutes")
    log.info("--- --- ---")
